## 134. How would you scale an LLM application from **100 to 100,000 users**?

> **Answer:** “I would make the application horizontally scalable and separate the **API, LLM, retrieval, and asynchronous workloads**. I would introduce load balancing, autoscaling, caching, queues, rate limiting, connection pooling, and observability.”

```text
100K Users
    ↓
CDN / API Gateway
    ↓
Load Balancer
    ↓
Stateless API Instances
    ↓
 ┌──────────┬───────────┐
 ↓          ↓           ↓
Cache     RAG/Search   Queue
 ↓          ↓           ↓
        LLM Gateway   Workers
             ↓
        Model Provider
```

**Key points:**
- Stateless FastAPI services
- Horizontal autoscaling
- Redis/cache
- Async queues for long-running workflows
- LLM rate-limit management
- Connection pooling
- Vector-search scaling
- Token/cost controls
- Monitoring and autoscaling

**One-liner:**

> **“At 100K users, I scale the application horizontally, decouple asynchronous workloads, cache aggressively, control LLM concurrency, and scale the retrieval and inference layers independently.”**

---

## 135. **Horizontal vs Vertical scaling?**

> **Answer:** “Vertical scaling means increasing the capacity of a single machine; horizontal scaling means adding more instances. For production AI applications, I generally prefer horizontal scaling because it provides better elasticity and availability.”

```text
Vertical:
1 Instance → Bigger Instance

Horizontal:
1 Instance → 10 Instances
```

| | Vertical | Horizontal |
|---|---|---|
| Approach | Bigger machine | More machines |
| Scalability | Limited | High |
| Availability | Lower | Higher |
| Cost flexibility | Limited | Better |
| AI API layer | Less preferred | Preferred |

**One-liner:**

> **“For stateless AI APIs, I prefer horizontal scaling; vertical scaling is mainly useful when a workload has resource constraints that cannot be distributed easily.”**

---

## 136. How do you handle **LLM rate limits**?

> **Answer:** “I implement rate-limit management using **concurrency limits, request throttling, exponential backoff, queues, and model fallback**. I also monitor provider-specific rate-limit headers and usage.”

```text
Requests
   ↓
Rate Limiter
   ↓
Concurrency Control
   ↓
LLM
   ↓
429?
 ├── No → Response
 └── Yes → Backoff / Queue / Fallback
```

### Controls
- Token-per-minute limits
- Requests-per-minute limits
- Per-user quotas
- Global concurrency limits
- Queue-based buffering
- Exponential backoff
- Model fallback

**One-liner:**

> **“I treat LLM rate limits as a capacity-management problem and control concurrency before requests reach the provider.”**

---

## 137. How do you implement **retries with exponential backoff**?

> **Answer:** “For transient failures such as 429 or 5xx responses, I retry a bounded number of times with exponentially increasing delays and jitter.”

```text
Attempt 1 → 1 sec
Attempt 2 → 2 sec
Attempt 3 → 4 sec
Attempt 4 → Fail / Fallback
```

Conceptually:

```python
delay = base * (2 ** attempt) + jitter
```

### Important

I **don't retry every error**.

```text
429 / 503 → Retry
Timeout   → Retry
400       → Don't retry
401/403   → Don't retry
Invalid request → Don't retry
```

**One-liner:**

> **“I use bounded exponential backoff with jitter for transient errors and avoid retrying permanent errors.”**

---

## 138. Where would you introduce **caching**?

> **Answer:** “I introduce caching at expensive and repeatable stages such as **LLM responses, embeddings, retrieval results, and frequently accessed application data**.”

```text
User
 ↓
API
 ↓
Cache?
 ├── HIT → Response
 └── MISS
      ↓
    RAG / LLM
      ↓
    Store
      ↓
   Response
```

### Common layers

```text
L1 → Application memory
L2 → Redis
L3 → Database / Search
L4 → LLM
```

**One-liner:**

> **“I cache deterministic or frequently repeated operations while carefully defining TTL and invalidation so stale AI responses aren't served.”**

---

## 139. What would you **cache in a RAG application**?

> **Answer:** “I would cache expensive intermediate results where the underlying data is relatively stable.”

### Good candidates

```text
1. Query embeddings
2. Retrieval results
3. Reranking results
4. LLM responses
5. Frequently accessed documents
6. Metadata/master data
```

```text
Query
 ↓
Embedding Cache
 ↓
Search
 ↓
Retrieval Cache
 ↓
Reranker
 ↓
LLM Response Cache
```

### Important

Cache keys should consider:

```text
query
model
prompt version
index/version
tenant
filters
```

**One-liner:**

> **“For RAG, I cache embeddings, retrieval results and selected LLM responses, but include tenant, filters, model and prompt/index versions in the cache key to prevent incorrect or stale responses.”**

---

## 140. How do you reduce **LLM latency**?

> **Answer:** “I reduce latency by minimizing input/output tokens, using smaller models where appropriate, parallelizing independent operations, streaming responses, caching repeated requests, and avoiding unnecessary LLM calls.”

```text
Latency Optimization
├── Smaller model
├── Fewer tokens
├── Streaming
├── Caching
├── Parallel tool calls
├── Async execution
├── Reduce RAG Top-K
└── Avoid unnecessary agent iterations
```

### Example

Instead of:

```text
Search → LLM → Tool → LLM → Tool → LLM
```

I parallelize independent operations:

```text
       ┌→ Search A ─┐
Query ─┼→ Search B ─┼→ LLM
       └→ Metadata ─┘
```

**One-liner:**

> **“I optimize LLM latency through token reduction, model routing, caching, streaming, parallel execution and reducing unnecessary model/tool calls.”**

---

## 141. How do you reduce **token consumption**?

> **Answer:** “I control both input and output tokens by reducing unnecessary context and enforcing output limits.”

### Techniques

```text
Large Context
    ↓
Metadata Filtering
    ↓
Top-K
    ↓
Reranking
    ↓
Context Compression
    ↓
LLM
```

- Smaller/relevant chunks
- Reranking
- Context compression
- Conversation summarization
- Remove duplicate context
- Limit tool output
- Structured output
- `max_tokens`
- Smaller models for simple tasks

**One-liner:**

> **“The biggest optimization is controlling input context; I retrieve only relevant information, compress it, remove duplicates, and constrain output tokens.”**

---

## 142. How do you reduce **embedding costs**?

> **Answer:** “I reduce embedding cost by embedding only what is necessary and avoiding repeated embedding of unchanged content.”

### Techniques

- Incremental ingestion
- Content hashing
- Deduplication
- Batch embedding
- Smaller/appropriate embedding model
- Cache embeddings
- Reuse existing vectors
- Avoid embedding irrelevant metadata

```text
Document
 ↓
Content Hash
 ↓
Changed?
 ├── No → Reuse Existing Embedding
 └── Yes → Generate Embedding
```

**One-liner:**

> **“I primarily control embedding cost through incremental indexing, content-hash-based change detection, deduplication, batching and embedding caching.”**

---

## 143. How do you optimize **vector search**?

> **Answer:** “I optimize vector search through appropriate indexing, metadata filtering, hybrid retrieval, dimensionality choices, Top-K tuning and reranking only a small candidate set.”

```text
Query
 ↓
Metadata Filter
 ↓
ANN Vector Search
 ↓
Top-20
 ↓
Reranker
 ↓
Top-5
```

### Techniques

- ANN indexing
- Metadata filtering
- Hybrid search
- Appropriate embedding dimension
- Top-K tuning
- Partitioning/sharding where supported
- Rerank only Top-N
- Index optimization
- Query/result caching

**One-liner:**

> **“I reduce vector-search latency by narrowing the search space with metadata filters, using ANN/hybrid retrieval, limiting candidate K, and applying expensive reranking only to the shortlisted results.”**

---

## 144. How do you handle **concurrent agent workflows**?

> **Answer:** “I make workflows asynchronous and isolate state per execution using a unique **workflow/thread ID**. Independent nodes can execute in parallel, while shared resources are protected through concurrency controls.”

```text
                Request
                   ↓
              Workflow ID
                   ↓
          ┌────────┼────────┐
          ↓        ↓        ↓
       Agent A  Agent B  Agent C
          ↓        ↓        ↓
          └────────┼────────┘
                   ↓
              Aggregator
```

### Controls

- Async execution
- Unique workflow/thread ID
- Stateless workers
- Queue-based execution
- Per-agent concurrency limits
- Checkpointing
- Idempotency
- Distributed locks where required

**One-liner:**

> **“I isolate each workflow using a unique state/checkpoint ID, execute independent tasks asynchronously, and use queues and concurrency controls to prevent resource contention.”**

---

## 145. How do you implement **backpressure**?

> **Answer:** “Backpressure prevents incoming requests from overwhelming downstream services such as the LLM, database, or vector store. I use bounded queues, concurrency limits, rate limiting and controlled rejection.”

```text
High Traffic
    ↓
API Gateway
    ↓
Rate Limiter
    ↓
Bounded Queue
    ↓
Worker Pool
    ↓
LLM
```

If the queue is full:

```text
Queue Full
   ↓
Reject / Throttle / Retry-After
```

### Controls

- Bounded queues
- Worker pools
- Semaphore/concurrency limits
- Rate limiting
- Load shedding
- Priority queues
- Autoscaling

**One-liner:**

> **“Backpressure ensures upstream traffic doesn't overwhelm downstream AI services by limiting concurrency and buffering work in controlled queues.”**

---

## 146. How do you design for **high availability**?

> **Answer:** “I eliminate single points of failure and deploy critical components across multiple availability zones with autoscaling, health checks, retries, failover and managed services.”

```text
                    Users
                      ↓
               Load Balancer
                /           \
               ↓             ↓
          API Instance   API Instance
               \             /
                ↓           ↓
                 Shared Services
                      ↓
              ┌───────┴────────┐
              ↓                ↓
           Search             LLM
              ↓                ↓
           Replica          Fallback
```

### Production controls

**Availability**
- Multi-zone deployment
- Multiple API instances
- Autoscaling
- Health checks
- Load balancing

**Reliability**
- Retries + exponential backoff
- Circuit breakers
- Idempotency
- Checkpointing
- Graceful degradation

**AI-specific**
- Model fallback
- Provider fallback where justified
- Cached responses
- Queue-based async processing

**Data**
- Replication
- Backups
- Disaster recovery
- Recovery Point Objective / Recovery Time Objective

**One-liner:**

> **“I design HA by removing single points of failure, using multi-zone stateless services, replicated data, autoscaling, health checks, retries, circuit breakers and LLM fallback, with defined RPO/RTO.”**